<a href="https://colab.research.google.com/github/BassemRamdan/AI-Resume-Intelligence/blob/main/EDA_and_Data_Splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Resume Intelligence - PDF Extraction, EDA and Data Splitting
This notebook downloads the resume PDF dataset from Hugging Face, extracts text from the PDFs, performs Exploratory Data Analysis (EDA), and splits the data into train, validation, and test sets.

In [ ]:
!pip install huggingface_hub pandas matplotlib seaborn scikit-learn pypdf tqdm datasets

In [ ]:
import os
import pandas as pd
from huggingface_hub import snapshot_download
from pypdf import PdfReader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

print("Downloading dataset files from Hugging Face...")
# Download the whole dataset repository locally in Colab
dataset_path = snapshot_download(repo_id="BassemRamdan/data", repo_type="dataset")
print(f"Dataset downloaded to: {dataset_path}")

In [ ]:
# Extract Text from PDFs and convert to Pandas DataFrame
data = []
# Find folders representing categories (ignoring hidden folders like .git)
categories = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d)) and not d.startswith('.git')]

print(f"Found {len(categories)} categories. Extracting text from PDFs...")

for category in tqdm(categories, desc="Categories"):
    cat_path = os.path.join(dataset_path, category)
    for filename in os.listdir(cat_path):
        if filename.lower().endswith('.pdf'):
            file_path = os.path.join(cat_path, filename)
            text = ""
            try:
                reader = PdfReader(file_path)
                for page in reader.pages:
                    extracted = page.extract_text()
                    if extracted:
                        text += extracted + " "
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
                
            # Only append if we got some text
            if text.strip():
                data.append({
                    "category": category,
                    "filename": filename,
                    "text": text.strip()
                })

df = pd.DataFrame(data)
print(f"\nDataset shape: {df.shape}")
df.head()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Basic information
print("\nDataset Info:")
df.info()

In [ ]:
# Visualize the distribution of resume categories
plt.figure(figsize=(12, 8))
sns.countplot(data=df, y='category', order=df['category'].value_counts().index)
plt.title('Distribution of Resume Categories')
plt.xlabel('Count')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

In [ ]:
# Add a column for text length to see the distribution of document sizes
df['text_length'] = df['text'].apply(len)

plt.figure(figsize=(10, 6))
sns.histplot(df['text_length'], bins=50)
plt.title('Distribution of Resume Text Lengths (Characters)')
plt.xlabel('Text Length')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Split the dataset into train, validation, and test sets (70% train, 15% val, 15% test)
print("Splitting data into Train, Validation, and Test sets...")

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['category'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['category'])

print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

In [ ]:
# Save the extracted texts to HuggingFace Dataset format (which is much better for training LLMs)
final_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df.reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df.reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df.reset_index(drop=True))
})

print(final_dataset)

# If you want to push this processed TEXT dataset back to Hugging Face so you don't have to extract PDFs again:
# final_dataset.push_to_hub("BassemRamdan/resume-text-dataset")